# G-reasoner on the team graph (`fin10k_v1`)

**How to use:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.

Where things live:
- **Your code and team data**: the repo on your Local (`~/FinGraphRAG/financial-graphrag`), synced to Drive by Google Drive for desktop and used read-only here. Edit on the Local, re-run the cell in Colab.

- **gfmrag working data and cache**: `My Drive/fin_greasoner_cache/`, so a new runtime skips the slow embedding step.

Success = the last cell prints `=== G-reasoner loaded ...` and ranked documents for each question.

## 1. Install Python 3.12
gfmrag needs Python 3.12, so it lives in its own venv at `/content/py312`, separate from Colab's own packages.

In [1]:
!apt-get update -qq
!apt-get install -y -qq python3.12 python3.12-venv
!python3.12 --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-pip-whl.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../python3-pip-whl_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Selecting previously unselected package python3-setuptools-whl.
Preparing to unpack .../python3-setuptools-whl_68.1.2-2ubuntu1.2_all.deb ...
Unpacking python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Selecting previously unselected package python3.12-venv.
Preparing to unpack .../python3.12-venv_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-venv (3.12.3-1ubuntu0.17) ...
Setting up python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Setting up python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Setting up python3.12-venv (3.12.3-1ubuntu0.17) ...
Python 3.12.

## 2. Create the venv and install gfmrag + ninja
Slow (several minutes). gfmrag is pinned to 2.0.0, the version the scripts were written against.

In [2]:
!python3.12 -m venv /content/py312
!/content/py312/bin/python -m pip install -q --upgrade pip
!/content/py312/bin/python -m pip install -q gfmrag==2.0.0 ninja
!/content/py312/bin/python -c "import gfmrag, ninja; print('gfmrag + ninja imported successfully')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO 11-25 02:50:29 [__init__.py:216] Automatically detected platform cuda.
gfmrag + ninja imported successfully


## 3. Put the venv on PATH
gfmrag steps run with the venv's Python (`PYTHON`). PATH must include the venv so ninja is found when G-reasoner compiles its CUDA kernel on first use.

In [3]:
import os, subprocess

PYTHON = "/content/py312/bin/python"
os.environ["PATH"] = "/content/py312/bin:" + os.environ["PATH"]
print("ninja:", subprocess.run(["which", "ninja"], capture_output=True, text=True).stdout)

ninja: /content/py312/bin/ninja



## 4. Patch gfmrag's PyG version parsing
gfmrag crashes on PyG versions like `2.8.0.post1`. This keeps only the first three parts.

In [4]:
from pathlib import Path

path = Path("/content/py312/lib/python3.12/site-packages/gfmrag/models/ultra/layers.py")
text = path.read_text()
old = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
new = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3]]'
if old in text:
    path.write_text(text.replace(old, new))
    print("Patch applied.")
elif new in text:
    print("Patch already applied.")
else:
    print("Original line not found. Check layers.py manually.")

Patch applied.


## 5. Mount Google Drive and find the repo
Colab asks for permission the first time. Your Mac syncs `~/FinGraphRAG` to Drive (Drive for desktop, "folders from your computer"), which Colab sees under `/content/drive/Othercomputers/<computer name>/`. The repo is used **read-only** from there: your scripts from `G-reasoner/`, team data from `data/`. Team data is whatever your Mac checkout has, so `git pull` on the Mac to get teammates' updates.

All gfmrag output goes to `My Drive/fin_greasoner_cache/`, never into the repo folder.

In [5]:
import glob
from google.colab import drive
drive.mount("/content/drive")

found = glob.glob("/content/drive/Othercomputers/*/FinGraphRAG/financial-graphrag")
print("synced repo:", found or "NOT FOUND")
assert len(found) == 1, "Check Drive for desktop: ~/FinGraphRAG must be synced (see 'Computers' in Drive on the web)."
REPO = found[0]
CODE = f"{REPO}/G-reasoner"

CACHE = "/content/drive/MyDrive/fin_greasoner_cache"  # gfmrag data root + results (survives runtime resets)
DATA  = f"{CACHE}/data"
os.makedirs(DATA, exist_ok=True)

need = [f"{CODE}/{f}" for f in ["prepare_data.py", "entity_index.py", "gfm_components.py", "index_graph.py", "query.py",
                                    "evaluate.py", "layer_trace.py"]]
need += [f"{REPO}/data/graph/fin10k_v1/processed/stage1/{f}" for f in ["nodes.csv", "relations.csv", "edges.csv"]]
missing = [p.replace(REPO + "/", "") for p in need if not os.path.exists(p)]
print("missing files:", missing or "none (if some are missing, Drive may still be uploading)")

Mounted at /content/drive
synced repo: ['/content/drive/Othercomputers/My Mac/FinGraphRAG/financial-graphrag']
missing files: none (if some are missing, Drive may still be uploading)


## 6. Prepare the graph
Copies `data/graph/fin10k_v1/processed/stage1` from the clone into `DATA` and writes `raw/documents.json`, which gfmrag requires even when stage1 exists. If the team graph changed since the last run, the old embedding cache is removed.

In [12]:
!python "{CODE}/prepare_data.py" --repo "{REPO}" --graph fin10k_v1 --root "{DATA}"

unchanged stage1 -> /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1/processed/stage1
wrote 1855 documents -> /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1/raw/documents.json


In [13]:
!nvidia-smi

Tue Nov 24 11:20:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 7. Embed the graph (stage2)
First run: downloads Qwen3-Embedding-0.6B and embeds all 3,841 nodes (a few minutes). Later runs find the cache on Drive and finish quickly. This runs on its own so the query step starts only one vLLM model on the T4.

In [14]:
!{PYTHON} "{CODE}/index_graph.py" --root "{DATA}" --data-name fin10k_v1

INFO 11-24 11:20:40 [__init__.py:216] Automatically detected platform cuda.
vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
stage2 ready in 0s: /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1/processed/stage2/71345f28ce99ab5be49b591c8c642cc6
nodes 3841, edges 31768 (with inverses), relations 38, feat_dim 1024
nodes by type: {'attribute': 1097, 'document': 1855, 'entity': 889}
documents loaded: 1855
files: ['graph.pt', 'node2id.json', 'rel2id.json']


## 8. Load G-reasoner and retrieve
Start nodes come from `entity_index.py` (alias matching on graph node names, no LLM). The first run also compiles G-reasoner's CUDA kernel, which takes a minute or two.

In [15]:
!{PYTHON} "{CODE}/query.py" --root "{DATA}" --data-name fin10k_v1 --top-k 5 \
    -q "What were Apple's total net sales in fiscal 2025?" \
    -q "How much DARZALEX revenue did J&J report in FY2025?" \
    -q "Which spent more on research and development in fiscal 2025, Apple or Microsoft?" \
    -q "Tell me about cybersecurity" \
    --out "{CACHE}/probe_results.jsonl"

INFO 11-24 11:20:54 [__init__.py:216] Automatically detected platform cuda.
vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
INFO 11-24 11:20:58 [utils.py:328] non-default args: {'task': 'embed', 'max_model_len': 8192, 'gpu_memory_utilization': 0.35, 'disable_log_stats': True, 'hf_overrides': {'is_matryoshka': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Embedding-0.6B'}
INFO 11-24 11:21:09 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
INFO 11-24 11:21:09 [config.py:708] Found sentence-transformers modules configuration.
INFO 11-24 11:21:09 [config.py:728] Found pooling configuration.
`torch_dtype` is deprecated! Use `dtype` instead!
WARNING 11-24 11:21:09 [__init__.py:2716] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 11-24 11:21:09 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 11-24 11:21:09 [__init__.py:1815] Using max model len 8192
IN

## 9. Score on the golden set
E2: zero-shot G-reasoner on golden set v2 (`vector_rag_baseline/data/questions_v2.jsonl`, 39 questions, all in `fin10k_v1`), scored with the vector baseline's own metrics (`vector_rag_baseline/rag/metrics.py`). Results go to `My Drive/fin_greasoner_cache/results/`, in the baseline's `per_question.jsonl` / `summary.json` format, then the baseline's `results_by_type.py` prints the table by question type.

`--layer-log` also writes `layers.jsonl`: node scores after every GNN layer (`layer_trace.py`), for the hop analysis (E7). Check the output for `WARNING` lines: a missing `fact_chunks` field (AllFacts falls back to Hit) or a layer readout that does not match the model.

In [ ]:
import time
RUN = f"{CACHE}/results/{time.strftime('%Y%m%d-%H%M%S')}"
!{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v1 --out "{RUN}" --layer-log 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL"
!cd "{REPO}/vector_rag_baseline" && {PYTHON} scripts/results_by_type.py "{RUN}"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
`torch_dtype` is deprecated! Use `dtype` instead!
W1121 17:59:08.557000 10188 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1121 17:59:08.557000 10188 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
=== G-reasoner loaded in 37s on cuda:0 ===
node types: {'attribute': 1097, 'document': 1855, 'entity': 889}
39 questions covered by the graph; skipped (gold not in graph): []
Load rspmm extension. This may take a while...
  v2_q01 [single] hit@5=1 recall=0.17  start=['Revenue', 'Apple Inc.', 'FY2025 (AAPL)']
  v2_q02 [single] hit@5=0 recall=0.00  start=['Research and development expense', 'Apple Inc.', 'FY2025 (AAPL)']
  v2_q03 [single] hit@5=0 recall=0.00  start=['Revenue', 'Apple Inc.', 'Services (AAPL)', 'FY2025 (AAPL)']
  v2_q04 [single] hit@5=0 recall=0.00  st

## 10. Diagnose retrieval
Runs the golden v2 questions four ways in one model load: all start nodes, without the Company/FiscalYear hub nodes, hubs only, and plain embedding similarity with no graph (`dense_qwen3`).

In [ ]:
import time
RUN = f"{CACHE}/results/diagnose-{time.strftime('%Y%m%d-%H%M%S')}"
!{PYTHON} "{CODE}/diagnose.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v1 --out "{RUN}" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING|NCCL|torch_dtype|cpp_extension"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
=== G-reasoner loaded in 38s on cuda:0 ===
node types: {'attribute': 1097, 'document': 1855, 'entity': 889}
39 covered questions; skipped []
Load rspmm extension. This may take a while...
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for "Apple's 10-K lists the countries where its outsourcing partners are mainly locat"; query runs without it
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for "Which company owns the patent families behind Johnson & Johnson's largest produc"; query runs without it
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for "Which product lines make up Apple's wireless headphones, and under which product"; query runs without it
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for 'Which countries does Apple na

In [16]:
!ls "{CODE}/configs" "{CODE}/prepare_sft.py" "{CODE}/synthetic/fin10k_v1"

'/content/drive/Othercomputers/My Mac/FinGraphRAG/financial-graphrag/G-reasoner/prepare_sft.py'

'/content/drive/Othercomputers/My Mac/FinGraphRAG/financial-graphrag/G-reasoner/configs':
finetune_fin10k.yaml

'/content/drive/Othercomputers/My Mac/FinGraphRAG/financial-graphrag/G-reasoner/synthetic/fin10k_v1':
golden_overlap.json  relation_dev_questions.jsonl  stats.json  train.json
relation_dev.json    review.md			   test.json


## 11. Fine-tuning data (E3)
Builds the dataset `fin10k_v1_sft` next to `fin10k_v1`: the same graph plus the synthetic pairs from `G-reasoner/synthetic/fin10k_v1/` (J&J + Apple to train, Microsoft to pick the best epoch). Relation pairs are repeated 5× so they are not drowned out. Then embeds the ~2,400 training questions (a few minutes, once).

Check the last lines: `train pairs: N of N kept by gfmrag`. If fewer are kept, gfmrag skipped some pairs; tell Claude.

In [17]:
!python "{CODE}/prepare_sft.py" --root "{DATA}" --synthetic "{CODE}/synthetic/fin10k_v1" --relation-repeat 5
!{PYTHON} "{CODE}/index_graph.py" --root "{DATA}" --data-name fin10k_v1_sft 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL|torch_dtype|cpp_extension"
import glob
done = glob.glob(f"{DATA}/fin10k_v1_sft/processed/stage2/*/train.pt")
assert done, "training questions were NOT processed (no train.pt): do not run section 12; send the output above to Claude"
print("OK: training questions processed ->", done[0])

/content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1_sft: train 2002 pairs (41 relation pairs x 5), test 377 pairs (Microsoft)
next: index_graph.py --data-name fin10k_v1_sft  (embeds the 2379 questions)
vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
stage2 ready in 1s: /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1_sft/processed/stage2/71345f28ce99ab5be49b591c8c642cc6
nodes 3841, edges 31768 (with inverses), relations 38, feat_dim 1024
nodes by type: {'attribute': 1097, 'document': 1855, 'entity': 889}
documents loaded: 1855
files: ['config.json', 'graph.pt', 'node2id.json', 'rel2id.json', 'test.pt', 'train.pt']
train pairs: 2002 of 2002 kept by gfmrag
test pairs: 377 of 377 kept by gfmrag
OK: training questions processed -> /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v1_sft/processed/stage2/71345f28ce99ab5be49b591c8c642cc6/train.pt


## 12. Fine-tune G-reasoner (E3)
Starts from G-reasoner-34M and trains on the synthetic pairs (`G-reasoner/configs/finetune_fin10k.yaml`: float32, because gfmrag's float16 training fails and the T4 has no bfloat16; lr 1e-4, 10 epochs, batch 8). After every epoch it scores the Microsoft pairs and keeps the best epoch, which is saved to `FT/pretrained/`.

**Run with `SMOKE = True` first** (1 epoch of 20 steps, about 2 minutes): it checks that training starts, fits in memory and saves a model. If it ends with `Training completed!`, set `SMOKE = False` and run the cell again for the real run (expect roughly 20-40 minutes; Colab must stay open).

If it stops with `CUDA out of memory`, add `trainer.args.train_batch_size=4` to `extra` and run again.

In [18]:
import time
SMOKE = False   # first run: True (quick check). Then False for the real training run.

FT = f"{CACHE}/finetune/{'smoke-' if SMOKE else ''}{time.strftime('%Y%m%d-%H%M%S')}"
extra = "trainer.args.num_epoch=1 trainer.args.max_steps_per_epoch=20" if SMOKE else ""
!cd /content && {PYTHON} -m gfmrag.workflow.sft_training --config-path "{CODE}/configs" --config-name finetune_fin10k \
    datasets.cfgs.root="{DATA}" hydra.run.dir="{FT}" {extra} 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL|torch_dtype|cpp_extension"
print("model saved:", os.path.exists(f"{FT}/pretrained/model.pth"), "->", FT)

[2026-11-24 11:30:49,231][__main__][INFO] - Config:
 seed: 1024
timeout: 60
save_pretrained: true
load_model_from_pretrained: rmanluo/G-reasoner-34M
datasets:
  _target_: gfmrag.graph_index_datasets.GraphIndexDataset
  cfgs:
    root: /content/drive/MyDrive/fin_greasoner_cache/data
    force_reload: false
    text_emb_model_cfgs: ${text_emb_model}
    use_node_feat: true
    use_relation_feat: true
    use_edge_feat: false
    inverse_relation_feat: text
  train_names:
  - fin10k_v1_sft
  valid_names:
  - fin10k_v1_sft
  init_datasets: true
  feat_dim: 1024
  max_datasets_in_memory: 2
  data_loading_workers: 2
model:
  _target_: gfmrag.models.gfm_reasoner.GraphReasoner
  use_ent_emb: early-late-fusion
  dtype: float32
  entity_model:
    _target_: gfmrag.models.ultra.models.QueryNBFNet
    input_dim: 1024
    hidden_dims:
    - 1024
    - 1024
    - 1024
    - 1024
    - 1024
    - 1024
    message_func: distmult
    aggregate_func: sum
    short_cut: true
    layer_norm: true
    retu

## 13. Score the fine-tuned model
Same scoring as section 9, on golden set v2 (E3), labelled `g-reasoner-ft`. Then the held-out relation check, for both models: `relation_dev` has 9 relation questions about facts that were **not** in training. Fine-tuned better than zero-shot there = it learned to follow relation links; better only on golden/trained facts = it memorised.

Uses the `FT` folder from section 12 (re-set it by hand if the runtime restarted).

In [19]:
assert os.path.exists(f"{FT}/pretrained/model.pth"), f"no trained model in {FT}: section 12 did not finish, so there is nothing to score"
RUN_FT = f"{FT}/eval_golden_v2"
!{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v1 --model "{FT}/pretrained" --out "{RUN_FT}" --layer-log 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL|torch_dtype|cpp_extension"
!cd "{REPO}/vector_rag_baseline" && {PYTHON} scripts/results_by_type.py "{RUN_FT}"

DEV = f"{CODE}/synthetic/fin10k_v1/relation_dev_questions.jsonl"
for name, model in [("zero-shot", "rmanluo/G-reasoner-34M"), ("fine-tuned", f"{FT}/pretrained")]:
    print(f"\n===== relation_dev, {name} =====")
    !{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v1 --model "{model}" --questions "{DEV}" --name "{name}" --out "{FT}/eval_relation_dev_{name}" 2>&1 | grep -E "^g-reasoner|^zero-shot|^fine-tuned|k=5|k=10"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
=== G-reasoner loaded in 37s on cuda:0 ===
node types: {'attribute': 1097, 'document': 1855, 'entity': 889}
39 questions covered by the graph; skipped (gold not in graph): []
Load rspmm extension. This may take a while...
  v2_q01 [single] hit@5=1 recall=0.83  start=['Revenue', 'Apple Inc.', 'FY2025 (AAPL)']
  v2_q02 [single] hit@5=1 recall=1.00  start=['Research and development expense', 'Apple Inc.', 'FY2025 (AAPL)']
  v2_q03 [single] hit@5=1 recall=1.00  start=['Revenue', 'Apple Inc.', 'Services (AAPL)', 'FY2025 (AAPL)']
  v2_q04 [single] hit@5=1 recall=0.60  start=['Microsoft Corporation', 'Revenue', 'FY2026 (MSFT)']
  v2_q05 [single] hit@5=1 recall=0.43  start=['Net income', 'Microsoft Corporation', 'FY2026 (MSFT)']
  v2_q06 [single] hit@5=1 recall=1.00  start=['Research and development expense', 'Microsoft Corporation', 'FY2026 (MSFT)']
  v2_q07 [single] hit@5=1 recall=1.00  start=['Research and development expense',

In [6]:
FT  = f"{CACHE}/finetune/20261124-113036"
HOP = f"{CACHE}/results/hop_case"
!{PYTHON} "{CODE}/hop_case.py" --repo "{REPO}" --root "{DATA}" --out "{HOP}/zero_shot_v2_q10.json" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL"
!{PYTHON} "{CODE}/hop_case.py" --repo "{REPO}" --root "{DATA}" --model "{FT}/pretrained" --out "{HOP}/finetuned_v2_q10.json" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING [0-9]|NCCL|torch_dtype|cpp_extension"
!/usr/bin/python3 "{CODE}/plot_hop_case.py" "{HOP}/zero_shot_v2_q10.json" "{HOP}/finetuned_v2_q10.json" --out "{HOP}/fig_hop_case_v2_q10"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
`torch_dtype` is deprecated! Use `dtype` instead!
W1125 02:54:44.192000 5017 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1125 02:54:44.192000 5017 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
=== G-reasoner loaded in 61s on cuda:0 ===
node types: {'attribute': 1097, 'document': 1855, 'entity': 889}
Load rspmm extension. This may take a while...
g-reasoner on v2_q10: start nodes ['Johnson & Johnson', 'Genmab A/S', 'Revenue', 'FY2025 (JNJ)']
node                                                       hops a/s  L0     L1     L2     L3     L4     L5     L6    
Genmab A/S                                                      0/0  2      2      1      1      1      1      1     
DARZALEX (JNJ)                                                  1/1  576    495    17